**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**06 | Sampling Sensitivity and Nested Training-Data Scaling**

![HARTH sampling sensitivity and nested training-data workflow](../../images/06_consolidated_workflow.svg)

**Objective:** Examine how label-independent per-participant sampling caps change activity coverage, and evaluate the effect of nested training-set expansion on Random Forest and weighted temporal CNN performance.

**Dataset:** Official NTNU AI Lab HARTH v2.0, with six lower-back and thigh acceleration channels sampled at 50 Hz. The Development cohort excludes S023 and S032 and retains eight documented activity codes.

**Experiments:** (A) A descriptive audit compares independent 250-, 500- and 1,000-window caps with full eligible source coverage; no model is fitted. (B) A separate fixed-Validation experiment compares Random Forest and weighted CNN trained on strictly nested 500- and 1,000-window-per-person sets. Independent sampling policies in (A) are not necessarily nested; the training cohorts in (B) are.

**Evaluation scope:** Participant-disjoint Development only. The same held-out Validation source rows are used across four model-size conditions. This notebook does not perform grouped cross-validation or reserved-participant assessment. Numerical findings require execution of the integrated notebook.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

Load source, sampling, statistical feature, model, plotting and Colab output utilities for the two complementary analyses.


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("wearable_activity_recognition")

# ----------------- 2. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

plot_style(font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11, legend_size=10, 
    figure_dpi=100, figure_size=(9, 4))

# ----------- 4. Machine Learning, Deep Learning Models and Metrics -----------
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score

# ------------------ 5. HARTH and Shared Research Utilities -------------------
from core.output_download import download_outputs_from_colab
from harth_data import (
    HARTH_SUBJECTS, SENSOR_COLUMNS, ACTIVITY_NAMES,
    prepare_harth_feature_cohort, prepare_harth_nested_window_cohort,
)
from core.sampling_audit import audit_temporal_sampling
from core.data_splitting import split_group_indices
from core.feature_engineering import extract_signal_features
from core.temporal_cnn import train_temporal_cnn
from core.ml_baselines import classification_class_report, evaluate_predictions_by_subject
from core.group_cross_validation import compare_models_by_subject

# ---------------------- 6. Device and Output Directory -----------------------
from core.computer_device import computer_device
device = computer_device()
DEVICE = device.type

output_dir = root / "wearable_activity_recognition" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
print("Research output directory:", output_dir)


**02. Reconstruct Full Eligible Development Windows**

Build 100-sample, label-pure and gap-free windows per participant, retaining source-window order. Extract 30 descriptive features for auditing only; no scaler or classifier is fitted.


In [ ]:
# ============= 02. Reconstruct Full Eligible Development Windows =============

# ----------------- 1. Define Development Cohort and Classes ------------------
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT_SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
TASK_CLASSES = tuple(range(1, 9))

# ------------------ 2. Prepare Full Eligible Source Windows ------------------
features, metadata, overview, excluded = prepare_harth_feature_cohort(
    DEVELOPMENT_SUBJECTS, root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES, window_size=100, step_size=50, sampling_rate=50,
)
assert set(metadata["subject_id"]) == set(DEVELOPMENT_SUBJECTS)
assert not (set(metadata["subject_id"]) & RESERVED_TEST)

# ------------------------ 3. Summarise Cohort Support ------------------------
print("Full eligible development windows:", len(metadata))
print("Feature columns (not used for fitting):", features.shape[1])

display(overview[["subject_id", "eligible_windows", "excluded_windows"]])


**03. Compare Independent Sampling Caps**

Independently select up to 250, 500 or 1,000 source windows per participant, evenly spaced in the eligible recording order, without consulting labels. Selection at a larger cap need not include every smaller-cap row.


In [ ]:
# =================== 03. Compare Independent Sampling Caps ===================

# --------------------- 1. Configure Label-Blind Sampling ---------------------
CAPS = (250, 500, 1000)
selected, per_subject, class_counts = audit_temporal_sampling(
    metadata=metadata,
    caps=CAPS,
    classes=TASK_CLASSES,
    forbidden_subjects=RESERVED_TEST,  # Exclude original reserved Test participants.
)

# ----------------------- 2. Compare Sampling Policies ------------------------
policies = ["full", "cap_250", "cap_500", "cap_1000"]
print("Window totals by policy:")

display(per_subject.groupby("policy")["windows"].sum().reindex(policies))
display(class_counts.pivot(index="label", columns="policy", values="windows")
        .reindex(index=TASK_CLASSES, columns=policies))


**04. Calculate Activity Retention**

Compute each class's retained-window percentage relative to the full eligible Development cohort. This measures source coverage, not classifier recall.


In [ ]:
# ===================== 04. Calculate Activity Retention ======================

# ----------------------- 1. Compute Activity Retention -----------------------
counts = class_counts.pivot(index="label", columns="policy", values="windows"
                           ).reindex(index=TASK_CLASSES, columns=policies)
retention = counts.div(counts["full"], axis=0).mul(100)  # Express retained class support as a percentage.
retention.index = [ACTIVITY_NAMES[k] for k in TASK_CLASSES]

# ------------------------ 2. Inspect Class Retention -------------------------
display(retention.round(2))

ax = retention[["cap_250", "cap_500", "cap_1000"]].plot(
    kind="bar", figsize=(12, 5), rot=35)
ax.set(title="HARTH class retention relative to full eligible development data",
       xlabel="Activity", ylabel="Retained windows (%)", ylim=(0, 105))
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
save_figure(ax.figure, output_dir / "06_sampling_class_retention.png")
plt.show()


**05. Measure Participant-Level Activity Support**

Compare the number of independent participants contributing each activity under the full and capped policies. Repeated windows from the same participant do not constitute independent support.


In [ ]:
# ============== 05. Measure Participant-Level Activity Support ===============

# ------------------- 1. Compare Participant-Level Support --------------------
support = (
    selected.groupby(["policy", "label"])["subject_id"].nunique()
    .rename("subjects").reset_index()
    .pivot(index="label", columns="policy", values="subjects")
    .reindex(index=TASK_CLASSES, columns=policies).fillna(0).astype(int)
)
support.index = [ACTIVITY_NAMES[k] for k in TASK_CLASSES]

# ---------------------- 2. Visualise Sampling Coverage -----------------------
display(support)

ax = support.plot(kind="bar", figsize=(12, 5), rot=35)
ax.set(title="Independent participant support by sampling policy",
       xlabel="Activity", ylabel="Participants", ylim=(0, 29))
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_figure(ax.figure, output_dir / "06_sampling_subject_support.png")
plt.show()


**06. Inspect Minority-Class Coverage**

Inspect Shuffling and the two stair classes by participant and policy, including any originally represented participant who loses all retained samples for a class.


In [ ]:
# ==================== 06. Inspect Minority-Class Coverage ====================

# ----------------------- 1. Select Minority Activities -----------------------
focus_codes = [3, 4, 5]
minority = selected.loc[selected["label"].isin(focus_codes)]

# --------------------- 2. Calculate Participant Coverage ---------------------
detail = pd.crosstab(
    index=[minority["subject_id"], minority["label"]],
    columns=minority["policy"],
).reindex(columns=policies, fill_value=0)
display(detail)

print("Source windows are sampled chronologically; no label balancing was applied.")


**07. Export Sampling-Audit Evidence**

Check source-row alignment, sampling-policy coverage and Test-participant exclusion. Save full cohort, excluded-code, retention, participant-support and selected source-row tables with the 06 sampling prefix.


In [ ]:
# ==================== 07. Export Sampling-Audit Evidence =====================

# ------------------- 1. Verify Cohort and Test Separation --------------------
assert set(selected["subject_id"]) == set(DEVELOPMENT_SUBJECTS)
assert not (set(selected["subject_id"]) & RESERVED_TEST)
assert len(selected.loc[selected["policy"] == "full"]) == len(metadata)
for cap in CAPS:
    policy_people = per_subject.loc[per_subject["policy"] == f"cap_{cap}"]
    assert set(policy_people["subject_id"]) == set(DEVELOPMENT_SUBJECTS)
    assert (policy_people["windows"] <= cap).all()

# ------------------- 2. Check Sampling and Class Integrity -------------------
assert all((retention <= 100 + 1e-8).to_numpy().ravel())
assert class_counts.groupby("policy")["windows"].sum().loc["full"] == len(metadata)

# ------------------ 3. Export Sampling Sensitivity Evidence ------------------
overview.to_csv(output_dir / "06_sampling_full_development_cohort_summary.csv", index=False)
excluded.to_csv(output_dir / "06_sampling_excluded_source_codes.csv", index=False)
per_subject.to_csv(output_dir / "06_sampling_policy_subject_totals.csv", index=False)
class_counts.to_csv(output_dir / "06_sampling_policy_class_counts.csv", index=False)
retention.rename_axis("activity").reset_index().to_csv(
    output_dir / "06_sampling_class_retention_percent.csv", index=False
)
support.rename_axis("activity").reset_index().to_csv(
    output_dir / "06_sampling_independent_subject_support.csv", index=False
)
detail.reset_index().to_csv(
    output_dir / "06_sampling_minority_activity_detail.csv", index=False
)
selected.to_csv(output_dir / "06_sampling_selected_window_metadata.csv", index=False)
# Release the full-feature matrix before constructing raw nested windows.
del features
print("Saved Development-only sampling audit outputs in:", output_dir)


**08. Build Strictly Nested Development Windows**

Construct two Training sampling levels where every 500-cap source window is retained inside the expanded 1,000-cap cohort. Extra windows are chosen without activity labels, preserving participant and source-row identity.


In [ ]:
# =============== 08. Build Strictly Nested Development Windows ===============

# ---------------------- 1. Build Nested Source Windows -----------------------
X, metadata, overview = prepare_harth_nested_window_cohort(
    subjects=DEVELOPMENT_SUBJECTS,
    data_dir=root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES,
    base_cap=500, expanded_cap=1000,  # Preserve cap-500 windows within the expanded cohort.
    window_size=100, step_size=50, sampling_rate=50,
)
assert set(metadata["subject_id"]) == set(DEVELOPMENT_SUBJECTS)
assert not (set(metadata["subject_id"]) & RESERVED_TEST)

# ------------------------ 2. Summarise Nested Cohort -------------------------
assert (overview["base_windows"] <= overview["expanded_windows"]).all()

print("Expanded source window array:", X.shape)

display(overview)


**09. Freeze Participant-Disjoint Validation**

Reproduce the fixed participant split using only the 500-cap cohort and seed 42. The same six Validation participants and identical 500-cap source rows are held out at both Train sizes.


In [ ]:
# ================ 09. Freeze Participant-Disjoint Validation =================

# ------------------- 1. Prepare Base Windows and Partition -------------------
base_mask = metadata["in_base_500"].to_numpy(dtype=bool)
base_row_ids = np.flatnonzero(base_mask)
original_parts = split_group_indices(
    metadata.iloc[base_row_ids]["subject_id"].to_numpy(),
    train_fraction=0.70, validation_fraction=0.20,
    test_fraction=0.10, random_state=42,
)

# ----------------------- 2. Verify Participant Groups ------------------------
train_people = set(metadata.iloc[base_row_ids[original_parts["train"]]]["subject_id"])
validation_people = set(metadata.iloc[base_row_ids[original_parts["validation"]]]["subject_id"])
unused_people = set(metadata.iloc[base_row_ids[original_parts["test"]]]["subject_id"])
assert train_people and validation_people and unused_people
assert not (train_people & validation_people or train_people & unused_people or validation_people & unused_people)
person_ids = metadata["subject_id"].to_numpy()

# --------------------- 3. Freeze Validation Source Rows ----------------------
validation_rows = np.flatnonzero(base_mask & np.isin(person_ids, list(validation_people)))  # Keep identical original Validation rows for both scales.

# ------------------- 4. Construct Nested Training Windows --------------------
base_train_rows = np.flatnonzero(base_mask & np.isin(person_ids, list(train_people)))
expanded_train_rows = np.flatnonzero(np.isin(person_ids, list(train_people)))
assert np.isin(base_train_rows, expanded_train_rows).all()
assert len(validation_rows) > 0
print("Train people:", sorted(train_people))
print("Validation people:", sorted(validation_people))
print("Train windows (base/expanded):", len(base_train_rows), len(expanded_train_rows))
print("Validation windows (fixed):", len(validation_rows))


**10. Inspect Added Training Class Support**

Compare class counts from base and expanded Train windows, separately from the fixed Validation class support.


In [ ]:
# ================= 10. Inspect Added Training Class Support ==================

# --------------------- 1. Compare Training Class Support ---------------------
train_class_counts = pd.DataFrame({
    "base_train": metadata.iloc[base_train_rows]["label"].value_counts(),
    "expanded_train": metadata.iloc[expanded_train_rows]["label"].value_counts(),
    "fixed_validation": metadata.iloc[validation_rows]["label"].value_counts(),
}).reindex(TASK_CLASSES).fillna(0).astype(int)

train_class_counts.index = [f"{code}: {ACTIVITY_NAMES[code]}" for code in TASK_CLASSES]
train_class_counts["new_train_windows"] = (
    train_class_counts["expanded_train"] - train_class_counts["base_train"]
)

# ---------------------- 2. Check Nested Data Expansion -----------------------
display(train_class_counts)

assert (train_class_counts["base_train"] > 0).all()
assert (train_class_counts["new_train_windows"] >= 0).all()


**11. Fit Random Forest at Two Training Sizes**

Fit two 120-tree Random Forest models on the same 30 deterministic features with class_weight balanced_subsample, min_samples_leaf three and seed 42. Only nested Train-set size changes.


In [ ]:
# ================ 11. Fit Random Forest at Two Training Sizes ================

# ---------------------- 1. Prepare Matched RF Features -----------------------
features = extract_signal_features(X, channel_names=SENSOR_COLUMNS)
y = metadata["label"].to_numpy()

# ------------------------ 2. Configure RF Conditions -------------------------
rf_predictions = {}

# ------------------------- 3. Fit RF Training Scales -------------------------
for name, train_rows in [("Train 500", base_train_rows), ("Train 1000", expanded_train_rows)]:
    forest = RandomForestClassifier(
        n_estimators=120, min_samples_leaf=3,
        class_weight="balanced_subsample", random_state=42, n_jobs=-1
    )
    forest.fit(features.iloc[train_rows], y[train_rows])
    rf_predictions[name] = forest.predict(features.iloc[validation_rows])
    print(name, "RF Validation Macro F1:",
          round(f1_score(y[validation_rows], rf_predictions[name],
                         labels=TASK_CLASSES, average="macro", zero_division=0), 4))


**12. Train Weighted CNNs at Two Sizes**

Train two CNNs with inverse-square-root Train-only loss weighting, Train-only channel scaling, five epochs, batch size 128 and learning rate 0.001. Increasing Train size also changes optimiser updates per epoch.


In [ ]:
# =================== 12. Train Weighted CNNs at Two Sizes ====================

# ------------------- 1. Configure Weighted CNN Experiments -------------------
cnn_predictions, histories = {}, {}


# ---------------------- 2. Train Nested CNN Conditions -----------------------
for name, train_rows in [("Train 500", base_train_rows), ("Train 1000", expanded_train_rows)]:
    print("Weighted CNN:", name)
    model, history, prediction, scaling = train_temporal_cnn(
        X_train=X[train_rows], y_train=y[train_rows],
        X_validation=X[validation_rows], y_validation=y[validation_rows],
        classes=TASK_CLASSES, epochs=5, batch_size=128,
        learning_rate=1e-3, seed=42, device=DEVICE,
        class_weighting="inverse_sqrt",
    )
    cnn_predictions[name] = prediction
    histories[name] = history
    display(history.round(4))


**13. Compare Fixed-Validation Model Metrics**

Compute fixed eight-class Macro F1 and balanced accuracy for all four model-size conditions on the same held-out source windows. Compare only this executed protocol, not historical output numbers.


In [ ]:
# ================ 13. Compare Fixed-Validation Model Metrics =================

# ------------------ 1. Compare Fixed Validation Predictions ------------------
prediction_sets = {
    "RF 500": rf_predictions["Train 500"],
    "RF 1000": rf_predictions["Train 1000"],
    "Weighted CNN 500": cnn_predictions["Train 500"],
    "Weighted CNN 1000": cnn_predictions["Train 1000"],
}
y_val = y[validation_rows]

# --------------------- 2. Calculate Matched Performance ----------------------
results = pd.DataFrame([
    {"model": name, "validation_macro_f1": f1_score(
         y_val, predicted, labels=TASK_CLASSES, average="macro", zero_division=0),
     "balanced_accuracy": balanced_accuracy_score(y_val, predicted),
     "validation_windows": len(y_val)}
    for name, predicted in prediction_sets.items()
])

display(results.round(4))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(results["model"], results["validation_macro_f1"])
ax.set(title="Training-data scaling on identical development Validation",
       ylabel="Macro F1", ylim=(0, 1))
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_figure(fig, output_dir / "06_nested_training_scale_macro_f1.png")
plt.show()


**14. Inspect Class and Participant-Level Effects**

Compare class-level precision, recall and F1 and participant-paired Macro F1 changes for 1,000 versus 500 Train windows, separately for Random Forest and weighted CNN.


In [ ]:
# ============== 14. Inspect Class and Participant-Level Effects ==============

# ----------------- 1. Prepare Class and Participant Results ------------------
class_tables = []
person_tables = []
val_people_ids = metadata.iloc[validation_rows]["subject_id"].to_numpy()

# ---------------------- 2. Evaluate Class-Level Changes ----------------------
for name, predicted in prediction_sets.items():
    class_result = classification_class_report(
        y_val, predicted, classes=TASK_CLASSES,
        class_names=[ACTIVITY_NAMES[c] for c in TASK_CLASSES]
    )
    class_result["model"] = name
    class_tables.append(class_result)
    person_tables.append(evaluate_predictions_by_subject(
        y_val, predicted, val_people_ids, TASK_CLASSES, name
    ))
classes_report = pd.concat(class_tables, ignore_index=True)
people_report = pd.concat(person_tables, ignore_index=True)

display(classes_report[["model","class","support","precision","recall","f1"]].round(4))

# ------------------- 3. Compare Paired Participant Scores --------------------
paired_tables, paired_summaries = [], []
for larger, smaller in [("RF 1000","RF 500"), ("Weighted CNN 1000","Weighted CNN 500")]:
    paired, summary = compare_models_by_subject(
        people_report, model_a=larger, model_b=smaller
    )
    display(pd.DataFrame([summary]).round(4))
    display(paired.round(4))
    paired_table = paired.copy()
    paired_table["model_a"] = larger
    paired_table["model_b"] = smaller
    paired_tables.append(paired_table)
    paired_summaries.append({"model_a": larger, "model_b": smaller, **summary})


**15. Export Nested Training-Scale Results**

Save nested source-window counts, four-condition results, class and participant metrics, paired differences, learning histories and source-aligned predictions under the 06 nested output family.


In [ ]:
# ================= 15. Export Nested Training-Scale Results ==================

# ---------------- 1. Verify Fixed Validation and Nested Data -----------------
assert len(base_train_rows) > 0
assert len(validation_rows) > 0
assert np.isin(base_train_rows, expanded_train_rows).all()

# ---------------------- 2. Check Participant Separation ----------------------
assert not (set(val_people_ids) & train_people)
assert not (set(val_people_ids) & RESERVED_TEST)
assert all(len(predicted) == len(y_val) for predicted in prediction_sets.values())
assert results["validation_macro_f1"].between(0, 1).all()

# ------------------- 3. Export Nested Training Comparison --------------------
overview.to_csv(output_dir / "06_nested_development_cohort_summary.csv", index=False)
train_class_counts.rename_axis("activity").reset_index().to_csv(
    output_dir / "06_nested_train_class_coverage.csv", index=False
)
results.to_csv(output_dir / "06_nested_validation_model_metrics.csv", index=False)
classes_report.to_csv(output_dir / "06_nested_validation_class_metrics.csv", index=False)
people_report.to_csv(output_dir / "06_nested_validation_subject_metrics.csv", index=False)
pd.concat(paired_tables, ignore_index=True).to_csv(
    output_dir / "06_nested_paired_subject_comparisons.csv", index=False
)
pd.DataFrame(paired_summaries).to_csv(
    output_dir / "06_nested_paired_summary.csv", index=False
)
for name, table in histories.items():
    safe_name = "500" if name == "Train 500" else "1000"
    table.to_csv(
        output_dir / f"06_nested_weighted_cnn_training_history_{safe_name}.csv", index=False
    )
pd.DataFrame({
    "source_row": np.tile(validation_rows, len(prediction_sets)),
    "subject_id": np.tile(val_people_ids, len(prediction_sets)),
    "true_source_code": np.tile(y_val, len(prediction_sets)),
    "model": np.repeat(list(prediction_sets.keys()), len(y_val)),
    "predicted_source_code": np.concatenate(list(prediction_sets.values())),
}).to_csv(output_dir / "06_nested_validation_predictions.csv", index=False)
print("Saved fixed-Validation nested training evidence in:", output_dir)

download_outputs_from_colab(folder="wearable_activity_recognition/output", stage="06")


**16. Scientific Interpretation and Research Boundaries**

**Sampling policy:** The independent 250-, 500- and 1,000-window cap audit describes class availability relative to the full eligible Development data and independent participant coverage. It fits no model; the cap-specific samples are not required to be nested.

**Nested training-scale comparison:** The 500-cap Train cohort is a strict source-row subset of the expanded 1,000-cap Train cohort. The four model conditions share the same held-out Validation source rows and participant identities. CNN epochs are fixed, not optimiser updates; class frequencies and effective update counts change with Train size.

**Limitations:** The evaluation uses one participant-disjoint Development split and a fixed seed. Overlapping windows, sparse minority classes and uneven participant-specific activity support limit generalisation claims. S023 and S032 are excluded. Three-fold grouped nested-training evaluation is a distinct experiment.

**Results:** Add numerical interpretation after the integrated notebook has executed and its stage-06 CSV and figure outputs have been inspected.
